In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

**Import Libraries**


In [ ]:
import random, math, time, warnings
import sympy as sp
import numpy as np
from scipy import integrate as sci
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")
random.seed(42); np.random.seed(42); torch.manual_seed(42)

x = sp.Symbol("x")

**Create Taylor functions**

In [ ]:
taylor_funcs = [
    sp.sin(x), sp.cos(x),
    sp.exp(x), sp.exp(-x),
    sp.log(1 + x),

    1 / (1 + x), 1 / (1 + x**2),
    sp.sqrt(1 + x),

    x**2 + x + 1,
    x**3 - x,
    (1 + x)**3,
    x**4 - x**2,

    sp.sinh(x), sp.cosh(x), sp.tanh(x), sp.atan(x),

    sp.sin(x) + sp.cos(x),
    sp.exp(-x) * sp.cos(x),
    sp.sin(x) * sp.cos(x),
    x**2 * sp.sin(x),

    (1 + x)**4
]
# coefficient variants
coeffs = [
    1,
    sp.Rational(1, 2), sp.Rational(1, 3),
    2, 3
]

taylor_data = []

for f in taylor_funcs:
    for c in coeffs:
        try:
            src_expr = sp.expand(c * f)

            # filter complex inputs
            if len(str(src_expr)) > 50:
                continue

            tgt_expr = sp.expand(
                sp.series(src_expr, x, 0, 5).removeO(),
                order='lex'
            )

            if tgt_expr == 0 or tgt_expr.is_number:
                continue

            if len(str(tgt_expr)) > 60:
                continue

            taylor_data.append({
                "src": str(src_expr),
                "tgt": str(tgt_expr)
            })

        except Exception:
            pass

**Dataset Generation**

In [ ]:
# dataset generation
random.seed(99)

for _ in range(5000):
    f1, f2 = random.sample(taylor_funcs, 2)
    c1, c2 = random.choice(coeffs), random.choice(coeffs)

    try:
        combo = sp.expand(c1 * f1 + c2 * f2)

        if len(str(combo)) > 50:
            continue

        expansion = sp.expand(
            sp.series(combo, x, 0, 5).removeO(),
            order='lex'
        )

        if expansion == 0 or expansion.is_number:
            continue

        if len(str(expansion)) > 60:
            continue

        taylor_data.append({
            "src": str(combo),
            "tgt": str(expansion)
        })

    except Exception:
        pass


print(f"Taylor pairs: {len(taylor_data)}")

# stable dedup
seen = set()
clean_data = []
for d in taylor_data:
    key = (d["src"], d["tgt"])
    if key not in seen:
        seen.add(key)
        clean_data.append(d)

taylor_data = clean_data

print(f"Taylor pairs (after dedup): {len(taylor_data)}")

Taylor pairs: 4952
Taylor pairs (after dedup): 3256


In [ ]:
_sym_bases = [
    1 + x, 1 + x**2, 1 + x + x**2,
    x * (1 - x), (1 - x)**2,

    sp.exp(x), sp.exp(-x), sp.exp(-2 * x),

    sp.sin(sp.pi * x),
    sp.sin(2 * sp.pi * x) + 1,
    sp.cos(sp.pi * x) + 1,

    1 / (1 + x), 1 / (1 + x**2),

    x**3 + 1,
    sp.log(1 + x) + 1
]
# lambdify and  normalize numerically
BASE_PDFS = []
for _sym in _sym_bases:
    try:
        _fn = sp.lambdify(x, _sym, "numpy")
        _I, _ = sci.quad(_fn, 0, 1)
        if _I > 1e-8 and np.isfinite(_I):
            # store as a normalized callable
            BASE_PDFS.append(lambda t, fn=_fn, I=_I: fn(t) / I)
    except Exception:
        pass

print(f"base PDFs: {len(BASE_PDFS)}")

base PDFs: 15


In [ ]:
def random_pdf():
    """Algorithm 2: combine 1-3 base PDFs with {+, *, /}, renormalize."""
    f = random.choice(BASE_PDFS)
    for _ in range(random.randint(0, 2)):
        b  = random.choice(BASE_PDFS)
        op = random.choice(["+", "*", "/"])
        w  = random.uniform(0.5, 2.0)          # scale parameter ω_k
        if   op == "+": combined = lambda t, f=f, b=b, w=w: f(t) + w * b(t)
        elif op == "*": combined = lambda t, f=f, b=b:       f(t) * b(t)
        else:           combined = lambda t, f=f, b=b:       f(t) / (b(t) + 0.1)
        try:
            I, _ = sci.quad(combined, 0, 1)
            if I > 1e-8 and np.isfinite(I):
                f = lambda t, g=combined, I=I: g(t) / I
            else:
                return None
        except Exception:
            return None
    return f

**Histogram Samples**

In [ ]:
def make_histogram(f, N, K):
    """
    Algorithm 1: integrate f over each bin to get bin probabilities,
    then draw N counts from a multinomial distribution.
    Returns (histogram array, expected counts array).
    """
    edges = np.linspace(0, 1, K + 1)
    try:
        probs = np.array([
            max(sci.quad(f, edges[i], edges[i + 1])[0], 0.0)
            for i in range(K)
        ])
    except Exception:
        return None, None
    if probs.sum() < 1e-8:
        return None, None
    probs /= probs.sum()
    return np.random.multinomial(N, probs), N * probs

In [ ]:
## build 200 histogram samples
hist_data = []
print("building histogram dataset...", end=" ", flush=True)
for _ in range(600):
    if len(hist_data) >= 200:
        break
    f = random_pdf()
    if f is None:
        continue
    K = random.randint(10, 30)
    N = random.randint(200, 1000)
    h, e = make_histogram(f, N, K)
    if h is not None:
        hist_data.append({"hist": h, "expected": e, "K": K, "N": N})
print(f"{len(hist_data)} samples")

building histogram dataset... 200 samples


**Tokenization**

In [ ]:

import re

def tokenize(expr):
    return re.findall(r'\d+/\d+|\d+|\*\*|[a-zA-Z_]+|[+\-*/(),]', expr)

# SPECIAL TOKENS
SPECIAL = ["<pad>", "<sos>", "<eos>", "<unk>"]

# BUILD VOCABULARY
tokens = set()
for d in taylor_data:
    tokens.update(tokenize(d["src"]))
    tokens.update(tokenize(d["tgt"]))

vocab = SPECIAL + sorted(tokens)

t2i = {tok: i for i, tok in enumerate(vocab)}
i2t = {i: tok for tok, i in t2i.items()}

PAD, SOS, EOS = t2i["<pad>"], t2i["<sos>"], t2i["<eos>"]

print(f"vocab size: {len(vocab)}")

vocab size: 155


In [ ]:
def encode(s):
    return [t2i.get(tok, t2i["<unk>"]) for tok in tokenize(s)]

def decode(ids):
    out = []
    for i in ids:
        tok = i2t.get(i, "")
        if tok == "<eos>":
            break
        if tok not in ("<pad>", "<sos>"):
            out.append(tok)
    return " ".join(out).replace(" / ", "/")

**Model Creation**

In [ ]:
class TaylorDS(Dataset):
    def __init__(self, indices):
        self.idx = indices

    def __len__(self):
        return len(self.idx)

    def __getitem__(self, i):
        d = taylor_data[self.idx[i]]

        src = torch.tensor(encode(d["src"]), dtype=torch.long)
        tgt = torch.tensor([SOS] + encode(d["tgt"]) + [EOS], dtype=torch.long)

        return src, tgt


def collate(batch):
    srcs, tgts = zip(*batch)

    return (
        pad_sequence(srcs, batch_first=True, padding_value=PAD),
        pad_sequence(tgts, batch_first=True, padding_value=PAD),
    )


#  reproducible split
all_idx = list(range(len(taylor_data)))
tr_idx, tmp = train_test_split(all_idx, test_size=0.2, random_state=42)
va_idx, te_idx = train_test_split(tmp, test_size=0.5, random_state=42)


#  DataLoaders (optimized)
train_dl = DataLoader(
    TaylorDS(tr_idx),
    batch_size=32,
    shuffle=True,
    collate_fn=collate,
    pin_memory=True
)

val_dl = DataLoader(
    TaylorDS(va_idx),
    batch_size=32,
    shuffle=False,
    collate_fn=collate,
    pin_memory=True
)

test_dl = DataLoader(
    TaylorDS(te_idx),
    batch_size=32,
    shuffle=False,
    collate_fn=collate,
    pin_memory=True
)

print(f"train: {len(tr_idx)}  val: {len(va_idx)}  test: {len(te_idx)}")

train: 2604  val: 326  test: 326


**LSTM Seq-to-seq model**

In [ ]:
class LSTMSeq2Seq(nn.Module):
    # Attention-based LSTM seq2seq.
    # Plain LSTM (single h,c bottleneck) fails on symbolic math — encoder
    # cannot compress a 30+ char expression into one vector.
    # Bahdanau attention lets the decoder look at ALL encoder hidden states.
    def __init__(self, vsz, emb=256, hid=512):
        super().__init__()
        self.vsz      = vsz
        self.hid      = hid
        self.enc_emb  = nn.Embedding(vsz, emb, padding_idx=PAD)
        self.enc_rnn  = nn.LSTM(emb, hid, batch_first=True, bidirectional=True,dropout=0.2)
        self.enc_proj = nn.Linear(hid * 2, hid)
        self.dec_emb  = nn.Embedding(vsz, emb, padding_idx=PAD)
        self.dec_rnn  = nn.LSTM(emb + hid, hid, batch_first=True,dropout=0.2)
        self.attn     = nn.Linear(hid * 2, 1)
        self.proj     = nn.Linear(hid, vsz)

    def _attend(self, dec_h, enc_out):
        dec_h_exp = dec_h.repeat(1, enc_out.size(1), 1)
        scores = self.attn(torch.tanh(torch.cat([dec_h_exp, enc_out], dim=-1)))
        weights   = torch.softmax(scores, dim=1)
        return (weights * enc_out).sum(dim=1, keepdim=True)

    def _encode(self, src):
        enc_out, (h, _) = self.enc_rnn(self.enc_emb(src))
        enc_out = self.enc_proj(enc_out)
        h = torch.tanh(self.enc_proj(
            torch.cat([h[0], h[1]], dim=-1)
        )).unsqueeze(0)
        return enc_out, h, torch.zeros_like(h)

    def forward(self, src, tgt, tf=0.5):
        enc_out, h, c = self._encode(src)
        B, T = tgt.shape
        outs = torch.zeros(B, T, self.vsz, device=src.device)
        tok  = tgt[:, 0]
        for step in range(1, T):
            emb     = self.dec_emb(tok.unsqueeze(1))
            context = self._attend(h.transpose(0, 1), enc_out)
            dec_in  = torch.cat([emb, context], dim=-1)
            out, (h, c) = self.dec_rnn(dec_in, (h, c))
            logits       = self.proj(out.squeeze(1))
            outs[:, step] = logits
            tok = tgt[:, step] if random.random() < tf else logits.argmax(1)
        return outs

    @torch.no_grad()
    def generate(self, src, maxlen=120):
        # greedy — used during training eval for speed
        self.eval()
        enc_out, h, c = self._encode(src)
        tok = torch.full((src.size(0),), SOS, dtype=torch.long, device=src.device)
        out = []
        for _ in range(maxlen):
            emb     = self.dec_emb(tok.unsqueeze(1))
            context = self._attend(h.transpose(0, 1), enc_out)
            e, (h, c) = self.dec_rnn(torch.cat([emb, context], dim=-1), (h, c))
            tok = self.proj(e.squeeze(1)).argmax(1)
            out.append(tok)
            if (tok == EOS).all():
                break
        return torch.stack(out, dim=1)

    @torch.no_grad()
    def generate_beam(self, src, beam_width=7, maxlen=120):
        # beam search for better sequence accuracy
        self.eval()
        assert src.size(0) == 1, "beam search: batch size must be 1"
        enc_out, h, c = self._encode(src)
        sequences = [([SOS], 0.0, h, c)]
        completed = []
        for _ in range(maxlen):
            all_cands = []
            for seq, score, h_, c_ in sequences:
                if seq[-1] == EOS:
                    completed.append((seq, score))
                    continue
                tok     = torch.tensor([[seq[-1]]], dtype=torch.long, device=src.device)
                emb     = self.dec_emb(tok)
                context = self._attend(h_.transpose(0, 1), enc_out)
                out, (h_new, c_new) = self.dec_rnn(torch.cat([emb, context], dim=-1), (h_, c_))
                lprobs  = torch.log_softmax(self.proj(out.squeeze(1)), dim=-1)
                topk    = torch.topk(lprobs, beam_width)
                for j in range(beam_width):
                    tok_id    = topk.indices[0][j].item()
                    length = len(seq) + 1
                    new_score = (score + topk.values[0][j].item()) / (length ** 0.6)
                    all_cands.append((seq + [tok_id], new_score, h_new, c_new))
            if not all_cands:
                break
            sequences = sorted(all_cands, key=lambda x: x[1], reverse=True)[:beam_width]

        candidates = completed if completed else sequences
        best = max(candidates, key=lambda x: x[1])[0]
        return torch.tensor(best[1:], device=src.device).unsqueeze(0)


**Positional Encoding**

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d, drop=0.1, maxlen=512):
        super().__init__()
        self.drop = nn.Dropout(drop)

        pe = torch.zeros(maxlen, d)
        pos = torch.arange(maxlen, dtype=torch.float32).unsqueeze(1)
        div = torch.exp(torch.arange(0, d, 2, dtype=torch.float32) * (-math.log(10000.0) / d))

        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)

        self.register_buffer("pe", pe.unsqueeze(0))  # (1, maxlen, d)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)].to(x.device)  #  ensure device match
        return self.drop(x)

**Transformer Seq-to-seq model**

In [ ]:
class TransformerSeq2Seq(nn.Module):
    def __init__(self, vsz, d=128, heads=4, enc_layers=3, dec_layers=3, ff=512, drop=0.1):
        super().__init__()
        self.vsz     = vsz
        self.d       = d
        self.src_emb = nn.Embedding(vsz, d, padding_idx=PAD)
        self.tgt_emb = nn.Embedding(vsz, d, padding_idx=PAD)
        self.dropout = nn.Dropout(drop)
        self.pe      = PositionalEncoding(d, drop)
        self.transformer = nn.Transformer(
            d, heads, enc_layers, dec_layers, ff, drop, batch_first=True
        )
        self.proj = nn.Linear(d, vsz)

    def _pad_mask(self, t):
        return t == PAD

    def _causal_mask(self, sz, dev):
        return torch.triu(torch.ones(sz, sz, device=dev), diagonal=1).bool()

    def forward(self, src, tgt):
        scale = math.sqrt(self.d)
        se = self.dropout(self.pe(self.src_emb(src) * scale))
        te = self.dropout(self.pe(self.tgt_emb(tgt) * scale))
        out = self.transformer(
            se, te,
            tgt_mask               = self._causal_mask(tgt.size(1), src.device),
            src_key_padding_mask   = self._pad_mask(src),
            tgt_key_padding_mask   = self._pad_mask(tgt),
            memory_key_padding_mask= self._pad_mask(src),
        )
        return self.proj(out)

    @torch.no_grad()
    def generate(self, src, maxlen=120):
        self.eval()
        scale = math.sqrt(self.d)
        s     = self.pe(self.src_emb(src) * scale)
        mem   = self.transformer.encoder(s, src_key_padding_mask=self._pad_mask(src))
        ys    = torch.full((src.size(0), 1), SOS, dtype=torch.long, device=src.device)
        for _ in range(maxlen):
            te  = self.pe(self.tgt_emb(ys) * scale)
            out = self.transformer.decoder(
                te, mem, tgt_mask=self._causal_mask(ys.size(1), src.device),
                tgt_key_padding_mask=self._pad_mask(ys)
            )
            nxt = self.proj(out[:, -1]).argmax(1, keepdim=True)
            ys  = torch.cat([ys, nxt], dim=1)
            if (nxt.squeeze(1) == EOS).all():
                break
        return ys[:, 1:]

    @torch.no_grad()
    def generate_beam(self, src, beam_width=7, maxlen=120):
        self.eval()
        scale = math.sqrt(self.d)

        s = self.pe(self.src_emb(src) * scale)
        mem = self.transformer.encoder(s, src_key_padding_mask=self._pad_mask(src))

        sequences = [([SOS], 0.0)]
        completed = []

        for _ in range(maxlen):
            all_cands = []

            for seq, score in sequences:
                if seq[-1] == EOS:
                    completed.append((seq, score))
                    continue

                ys = torch.tensor(seq, device=src.device).unsqueeze(0)
                te = self.pe(self.tgt_emb(ys) * scale)

                out = self.transformer.decoder(
                    te, mem,
                    tgt_mask=self._causal_mask(ys.size(1), src.device),
                    tgt_key_padding_mask=self._pad_mask(ys)
                )

                lprobs = torch.log_softmax(self.proj(out[:, -1]), dim=-1)
                topk = torch.topk(lprobs, beam_width)

                for j in range(beam_width):
                    tok = topk.indices[0][j].item()
                    length = len(seq) + 1
                    new_score = (score + topk.values[0][j].item()) / (length ** 0.6)
                    all_cands.append((seq + [tok], new_score))

            if not all_cands:
                break

            sequences = sorted(all_cands, key=lambda x: x[1], reverse=True)[:beam_width]

        candidates = completed if completed else sequences
        best = max(candidates, key=lambda x: x[1])[0]
        return torch.tensor(best[1:], device=src.device).unsqueeze(0)


**Model Training**

In [ ]:
def run_epoch(model, dl, opt, criterion, device, is_lstm, training=True, tf=0.5):
    model.train() if training else model.eval()

    total = 0.0
    ctx = torch.enable_grad() if training else torch.no_grad()

    with ctx:
        for src, tgt in dl:
            src, tgt = src.to(device), tgt.to(device)

            if is_lstm:
                out = model(src, tgt, tf=tf if training else 0.0)
                logits = out[:, 1:].reshape(-1, out.shape[-1])
                labels = tgt[:, 1:].reshape(-1)
            else:
                out = model(src, tgt[:, :-1])
                logits = out.reshape(-1, out.shape[-1])
                labels = tgt[:, 1:].reshape(-1)

            loss = criterion(logits, labels)

            if training:
                opt.zero_grad(set_to_none=True)
                loss.backward()

                #  gradient clipping
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

                opt.step()

            total += loss.item()

    return total / max(1, len(dl))

In [ ]:
def fit(model, is_lstm, tag, epochs=100):
    lr = 3e-4 if is_lstm else 5e-4

    opt = optim.Adam(
        model.parameters(),
        lr=lr,
        betas=(0.9, 0.999 if is_lstm else 0.98)
    )

    sched = optim.lr_scheduler.ReduceLROnPlateau(
        opt, patience=8, factor=0.5, min_lr=1e-5
    )

    best_loss = float("inf")
    tr_log, vl_log = [], []
    t0 = time.time()

    for ep in range(1, epochs + 1):

        #  Teacher forcing decay
        tf = max(0.1, 0.5 * (1 - 2 * ep / epochs))

        #  Transformer warmup
        if not is_lstm and ep <= 5:
            for g in opt.param_groups:
                g['lr'] = lr * (ep / 5)

        # training
        tr = run_epoch(
            model, train_dl, opt, criterion,
            device, is_lstm, training=True, tf=tf
        )

        # validation
        vl = run_epoch(
            model, val_dl, opt, criterion,
            device, is_lstm, training=False
        )

        sched.step(vl)

        tr_log.append(tr)
        vl_log.append(vl)

        #  stable checkpoint save
        if vl < best_loss - 1e-4:
            best_loss = vl
            torch.save(model.state_dict(), f"/tmp/{tag}.pt")

        if ep % 10 == 0:
            print(f"  ep {ep}/{epochs}  tr={tr:.3f}  vl={vl:.3f}  tf={tf:.2f}  ppl={math.exp(vl):.1f}")

    #  load best model
    model.load_state_dict(torch.load(f"/tmp/{tag}.pt", map_location=device))

    print(f"  done in {time.time()-t0:.0f}s  best_val={best_loss:.4f}")

    return tr_log, vl_log

In [ ]:
def safe_parse(expr):
    expr = expr.replace(" ", "")
    expr = expr.replace("+-", "-").replace("-+", "-")
    expr = expr.replace("--", "+").replace("++", "+")
    expr = expr.replace("*+", "*").replace("+*", "*")
    expr = expr.replace("-*", "-1*").replace("+*", "+1*")
    return sp.sympify(expr)

**Model Evaluation Metrics**

In [ ]:
def token_accuracy(model, dl, device, is_lstm):
    """Fraction of non-PAD tokens predicted correctly."""
    model.eval()
    ok = n = 0
    with torch.no_grad():
        for src, tgt in dl:
            src, tgt = src.to(device), tgt.to(device)
            if is_lstm:
                out  = model(src, tgt, tf=0.0)
                pred = out[:, 1:].argmax(-1)
                ref  = tgt[:, 1:]
            else:
                out  = model(src, tgt[:, :-1])
                pred = out.argmax(-1)
                ref  = tgt[:, 1:]
            mask = ref != PAD
            ok  += (pred[mask] == ref[mask]).sum().item()
            n   += mask.sum().item()
    return ok / n if n else 0.0

In [ ]:
def seq_accuracy(model, dl, device, use_beam=True):
    model.eval()
    ok = n = 0

    with torch.no_grad():
        for src, tgt in dl:
            src, tgt = src.to(device), tgt.to(device)

            for i in range(tgt.size(0)):
                src_i = src[i].unsqueeze(0)

                if use_beam and hasattr(model, "generate_beam"):
                    pred = model.generate_beam(src_i, beam_width=7)
                else:
                    pred = model.generate(src_i)

                ref_str = decode(tgt[i].tolist())
                hyp_str = decode(pred[0].tolist())

                try:
                    ref = safe_parse(ref_str)
                    hyp = safe_parse(hyp_str)

                    if sp.simplify(ref - hyp) == 0:
                        ok += 1
                    else:
                        f_ref = sp.lambdify(x, ref, "numpy")
                        f_hyp = sp.lambdify(x, hyp, "numpy")

                        pts = np.linspace(-1, 1, 5)

                        valid = True
                        for p in pts:
                            try:
                                if not np.isfinite(p):
                                    continue
                                if abs(f_ref(p) - f_hyp(p)) > 1e-3:
                                    valid = False
                                    break
                            except:
                                valid = False
                                break

                        if valid:
                            ok += 1

                except:
                    pass

                n += 1

    return ok / n if n else 0.0

In [ ]:
def chi2_gof(observed, expected, min_count=5):
    """
    Section 2.3 of the paper.
    X = Σ (N_k - n_k)² / n_k
    Good fit: X/ndf ≈ 1. Bins with expected count < 5 are excluded.
    """
    Nk = np.array(observed, dtype=float)
    nk = np.array(expected, dtype=float)
    mask = nk >= min_count
    if not mask.any():
        return None, None
    X   = np.sum((Nk[mask] - nk[mask])**2 / nk[mask])
    ndf = int(mask.sum())
    return float(X), ndf
print("\nchi-squared GOF check (5 samples):")
for hd in hist_data[:5]:
    X, ndf = chi2_gof(hd["hist"], hd["expected"])
    if X and ndf:
        print(f"  X={X:.2f}  ndf={ndf}  X/ndf={X/ndf:.3f}  ({'good' if 0.5 < X/ndf < 1.5 else 'check'})")


chi-squared GOF check (5 samples):
  X=9.90  ndf=8  X/ndf=1.238  (good)
  X=15.62  ndf=21  X/ndf=0.744  (good)
  X=25.18  ndf=25  X/ndf=1.007  (good)
  X=7.69  ndf=10  X/ndf=0.769  (good)
  X=10.89  ndf=11  X/ndf=0.990  (good)


In [ ]:
device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
VSIZ      = len(vocab)
LSTM_EP = 40
TF_EP   = 80
criterion = nn.CrossEntropyLoss(ignore_index=PAD)
print(f"\ndevice={device}  vocab={VSIZ}  lstm_epochs={LSTM_EP}  tf_epochs={TF_EP}\n")

print("[LSTM]")
lstm = LSTMSeq2Seq(VSIZ).to(device)
lstm_tr, lstm_vl = fit(lstm, is_lstm=True, tag="lstm",epochs=LSTM_EP)

print("\n[Transformer]")
tf = TransformerSeq2Seq(VSIZ, d=384, heads=12).to(device)
tf_tr, tf_vl = fit(tf, is_lstm=False, tag="tf",epochs=TF_EP)




device=cuda  vocab=155  lstm_epochs=40  tf_epochs=80

[LSTM]
  ep 10/40  tr=0.877  vl=1.284  tf=0.25  ppl=3.6
  ep 20/40  tr=0.452  vl=1.015  tf=0.10  ppl=2.8
  ep 30/40  tr=0.182  vl=0.908  tf=0.10  ppl=2.5
  ep 40/40  tr=0.058  vl=0.936  tf=0.10  ppl=2.5
  done in 206s  best_val=0.7891

[Transformer]
  ep 10/80  tr=0.493  vl=0.482  tf=0.38  ppl=1.6
  ep 20/80  tr=0.307  vl=0.355  tf=0.25  ppl=1.4
  ep 30/80  tr=0.218  vl=0.317  tf=0.12  ppl=1.4
  ep 40/80  tr=0.174  vl=0.305  tf=0.10  ppl=1.4
  ep 50/80  tr=0.090  vl=0.282  tf=0.10  ppl=1.3
  ep 60/80  tr=0.071  vl=0.306  tf=0.10  ppl=1.4
  ep 70/80  tr=0.041  vl=0.317  tf=0.10  ppl=1.4
  ep 80/80  tr=0.030  vl=0.316  tf=0.10  ppl=1.4
  done in 216s  best_val=0.2754


**Model Comparison**

In [ ]:
print(f"\n{'model':14} {'split':5} {'loss':>7} {'ppl':>8} {'tok%':>7} {'seq%':>7}")
print("-" * 52)

for split, dl in [("val", val_dl), ("test", test_dl)]:
    for model, is_lstm, name in [(lstm, True, "LSTM"), (tf, False, "Transformer")]:
        loss = run_epoch(model, dl, None, criterion, device, is_lstm, training=False)

        print(f"{name:14} {split:5} {loss:7.4f} {math.exp(loss):8.2f} "
              f"{token_accuracy(model, dl, device, is_lstm)*100:6.1f}% "
              f"{seq_accuracy(model, dl, device, use_beam=True)*100:6.1f}%")


model          split    loss      ppl    tok%    seq%
----------------------------------------------------
LSTM           val    0.7891     2.20   80.6%    2.5%
Transformer    val    0.2754     1.32   92.0%   13.8%
LSTM           test   0.8472     2.33   80.3%    3.4%
Transformer    test   0.2922     1.34   92.3%    8.0%


**Sample Results**

In [ ]:
print("\nsample predictions (3 test examples):")

_ex  = [taylor_data[i] for i in te_idx[:3]]
_src = pad_sequence(
    [torch.tensor(encode(d["src"])) for d in _ex],
    batch_first=True, padding_value=PAD
).to(device)

for model, name in [(lstm, "LSTM"), (tf, "Transformer")]:
    preds = []

    for i in range(_src.size(0)):
        src_i = _src[i].unsqueeze(0)

        if hasattr(model, "generate_beam"):
            pred = model.generate_beam(src_i)
        else:
            pred = model.generate(src_i)

        preds.append(pred[0])

    print(f"\n  {name}:")

    for i, d in enumerate(_ex):
        hyp = decode(preds[i].tolist())

        try:
            diff = sp.simplify(safe_parse(hyp) - safe_parse(d["tgt"]))
            tick = "✓" if diff == 0 else "✗"
        except:
            tick = "✗"

        print(f"    [{tick}] {d['src']}")
        print(f"         ref: {d['tgt']}")
        print(f"         got: {hyp}")


sample predictions (3 test examples):

  LSTM:
    [✗] 3*exp(x) + cosh(x)/3
         ref: 5*x**4/36 + x**3/2 + 5*x**2/3 + 3*x + 10/3
         got: 3 * x ** 4/36 + x ** 3/2 + 3 * x ** 2/3 + 10 * x + 10/3
    [✗] 3*x**2*sin(x) + 2*sin(x)*cos(x)
         ref: 5*x**3/3 + 2*x
         got: - * x ** 3/3 + 2 * x
    [✗] x**2 + x + 3*sin(x) + 1
         ref: -x**3/2 + x**2 + 4*x + 1
         got: ** x ** 3/2 + x ** 2 + 4 * x + 1

  Transformer:
    [✗] 3*exp(x) + cosh(x)/3
         ref: 5*x**4/36 + x**3/2 + 5*x**2/3 + 3*x + 10/3
         got: 11 * x ** 4/72 + x ** 3/18 + 7 * x ** 2/6 + x/3 + 10/3
    [✗] 3*x**2*sin(x) + 2*sin(x)*cos(x)
         ref: 5*x**3/3 + 2*x
         got: x ** 4/8 + 2 * x ** 3 - 3 * x ** 2/2 + 3
    [✗] x**2 + x + 3*sin(x) + 1
         ref: -x**3/2 + x**2 + 4*x + 1
         got: 3 * x ** 3/2 + 3 * x ** 2 + 4 * x + 1


**Data Visualisation**


In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(14, 4))

# loss curves
axs[0].plot(range(1, len(lstm_tr)+1), lstm_tr, "#3b82f6", lw=1.5, label="LSTM train")
axs[0].plot(range(1, len(lstm_vl)+1), lstm_vl, "#3b82f6", lw=1.5, ls="--", label="LSTM val")
axs[0].plot(range(1, len(tf_tr)+1), tf_tr,   "#f97316", lw=1.5, label="TF train")
axs[0].plot(range(1, len(tf_vl)+1), tf_vl,   "#f97316", lw=1.5, ls="--", label="TF val")
axs[0].set(xlabel="epoch", ylabel="CE loss", title="loss")
axs[0].legend(fontsize=8); axs[0].grid(alpha=0.2)

# perplexity
axs[1].plot(range(1, len(lstm_vl)+1), [math.exp(l) for l in lstm_vl], "#3b82f6", lw=1.5, label="LSTM")
axs[1].plot(range(1, len(tf_vl)+1), [math.exp(l) for l in tf_vl],   "#f97316", lw=1.5, label="TF")
axs[1].set(xlabel="epoch", ylabel="perplexity", title="val perplexity")
axs[1].legend(fontsize=8); axs[1].grid(alpha=0.2)

# chi-squared GOF distribution
ratios = []
for hd in hist_data:
    X, ndf = chi2_gof(hd["hist"], hd["expected"])
    if X and ndf and ndf > 0:
        ratios.append(X / ndf)
axs[2].hist(ratios, bins=20, color="#64748b", edgecolor="white", alpha=0.85)
axs[2].axvline(1.0, color="red", ls="--", lw=1.2, label="ideal=1")
axs[2].set(xlabel="X/ndf", ylabel="count", title="χ² goodness-of-fit")
axs[2].legend(fontsize=8); axs[2].grid(alpha=0.2)

plt.tight_layout()
plt.savefig("faseroh_results.png", dpi=150, bbox_inches="tight")
plt.close()
print("\nsaved faseroh_results.png")


saved faseroh_results.png
